# Eksploracyjna analiza danych — `viral_songs.csv`

## 1. Przygotowanie

### Importy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

### Wczytanie danych

In [ ]:
df = pd.read_csv('/Users/majazontek/Desktop/UCZELNIA/3 rok/SEMINARIUM LICENCJACKIE/MACHINE LEARNING/dane/viral_songs.csv')  # podmień ścieżkę

FEATURES = [
    "duration_sec", "danceability", "energy", "valence", "tempo_bpm",
    "acousticness", "instrumentalness", "speechiness", "liveness", "loudness_db",
]

# viral jako 0/1, log10(streams) do wykresów i korelacji
df["viral"] = df["viral"].astype(int)
df["log_streams"] = np.log10(df["streams"] + 1)

### Przegląd danych

In [ ]:
print(df.shape)
df.info()
display(df.head())
display(df[FEATURES + ["streams"]].describe().T)
print("Braki danych:\n", df.isna().sum())
print("Duplikaty:", df.duplicated().sum())

### Funkcja pomocnicza: histogram z nałożonym rozkładem normalnym

In [ ]:
def hist_with_normal(ax, x, name, bins=30):
    x = pd.Series(x).dropna()
    mean, var, std = x.mean(), x.var(), x.std()

    ax.hist(x, bins=bins, density=True, edgecolor="black")
    grid = np.linspace(x.min(), x.max(), 300)
    ax.plot(grid, stats.norm.pdf(grid, mean, std), color="red", label="N(μ, σ²)")
    ax.axvline(mean, color="black", ls="--", label="średnia")

    ax.set_title(f"{name}\nśrednia = {mean:.3g}   wariancja = {var:.3g}")
    ax.set_ylabel("Gęstość")
    ax.legend()

## 2. Zmienne objaśniające

### Histogramy

Dla każdej cechy histogram z nałożoną gęstością rozkładu normalnego o tej samej średniej i wariancji.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 22))
for ax, col in zip(axes.flat, FEATURES):
    hist_with_normal(ax, df[col], col)
plt.tight_layout()
plt.show()

## 3. Zmienne objaśniane

### `streams` — model 1

Histogram w oryginalnej skali i po przekształceniu `log10`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hist_with_normal(axes[0], df["streams"], "streams")
hist_with_normal(axes[1], df["log_streams"], "log10(streams + 1)")
plt.tight_layout()
plt.show()

### `viral` — model 2

In [ ]:
counts = df["viral"].value_counts().sort_index()
pct = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(["Nie viral (0)", "Viral (1)"], counts.values)
for bar, c, p in zip(bars, counts.values, pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{c}\n({p:.1f}%)",
            ha="center", va="bottom")
ax.set_title("Rozkład zmiennej viral")
ax.set_ylabel("Liczba piosenek")
ax.margins(y=0.15)
plt.show()

### `genre` — model 3

In [ ]:
genre_counts = df["genre"].value_counts()
genre_pct = genre_counts / genre_counts.sum() * 100

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(genre_counts.index, genre_counts.values)
for bar, c, p in zip(bars, genre_counts.values, genre_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{c}\n({p:.1f}%)",
            ha="center", va="bottom")
ax.set_title("Liczba piosenek według gatunku")
ax.set_ylabel("Liczba piosenek")
ax.tick_params(axis="x", rotation=45)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

## 4. Zależności między zmiennymi

### Korelacje Pearsona: cechy, `log10(streams)` i `viral`

Korzystamy z `log10(streams)`, bo przy `streams` współczynnik Pearsona byłby zaniżony i zdominowany przez kilka utworów z setkami milionów odtworzeń.

In [ ]:
cols = FEATURES + ["log_streams", "viral"]
corr_all = df[cols].corr()
mask = np.triu(np.ones_like(corr_all, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_all, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Korelacje cech audio z log10(streams) i viral")
plt.tight_layout()
plt.show()

### Korelacje Spearmana: cechy, `streams` i `viral`

Korelacja Spearmana liczona na rangach, więc można ją policzyć na oryginalnych `streams`.

In [ ]:
cols = FEATURES + ["streams", "viral"]
corr_all = df[cols].corr(method="spearman")
mask = np.triu(np.ones_like(corr_all, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_all, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Korelacje Spearmana cech audio ze streams i viral")
plt.tight_layout()
plt.show()

### Czy `streams` i `viral` są ze sobą powiązane?

In [ ]:
#korelacja
display(df[["streams", "log_streams", "viral"]].corr().round(3))

#statystyki opisowe
display(df.groupby("viral")["streams"].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="viral", y="log_streams", ax=axes[0])
axes[0].set_xticks([0, 1], ["Nie viral", "Viral"])
axes[0].set_title("log(streams) według viral")

bins = np.linspace(df["log_streams"].min(), df["log_streams"].max(), 40)
for v, label in [(0, "Nie viral"), (1, "Viral")]:
    axes[1].hist(df.loc[df["viral"] == v, "log_streams"], bins=bins,
                 density=True, alpha=0.6, label=label)
axes[1].set_xlabel("log10(streams + 1)")
axes[1].set_ylabel("Gęstość")
axes[1].set_title("Rozkład log(streams) według viral")
axes[1].legend()

plt.tight_layout()
plt.show()

### Od ilu odtworzeń piosenka jest viralem?

In [ ]:
print("Min streams wśród viral:    ", df.loc[df["viral"] == 1, "streams"].min())
print("Max streams wśród nie-viral:", df.loc[df["viral"] == 0, "streams"].max())

display(df.groupby("viral")["streams"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack())

### Gatunek a `viral`

In [ ]:
ct = pd.crosstab(df["genre"], df["viral"]).reindex(columns=[0, 1], fill_value=0)
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]
total = ct.sum(axis=1)
viral_pct = ct[1] / total * 100

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(ct.index, ct[0], label="Nie viral")
ax.bar(ct.index, ct[1], bottom=ct[0], label="Viral")
for i, (t, p) in enumerate(zip(total, viral_pct)):
    ax.text(i, t, f"{p:.1f}%", ha="center", va="bottom")
ax.set_title("Liczba piosenek według gatunku z udziałem viral")
ax.set_ylabel("Liczba piosenek")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

### Odsetek viral według gatunku i roku wydania

In [ ]:
viral_rate = pd.crosstab(df["genre"], df["release_year"], values=df["viral"], aggfunc="mean") * 100
viral_rate = viral_rate.loc[viral_rate.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(viral_rate, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% viral"}, ax=ax)
ax.set_title("Odsetek piosenek viralowych według gatunku i roku wydania (%)")
ax.set_xlabel("Rok wydania")
ax.set_ylabel("Gatunek")
plt.tight_layout()
plt.show()